In [2]:
import json
import joblib
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

from benchmark.config import (
    DATA_PROCESSED,
    DATASETS,
    MODELS_DIR,
    N_TRIALS,
    RESULTS_BENCHMARK,
)
from benchmark.models.random_forest_model import fit_random_forest_cross_validation
from benchmark.models.random_forest_model import fit_best_random_forest

In [3]:
data_name = "abalone"
cfg = DATASETS[data_name]
processed = DATA_PROCESSED / data_name

train_df = pd.read_csv(processed / "train.csv")
test_df = pd.read_csv(processed / "test.csv")

with open(processed / "metadata.json", "r") as f:
    metadata = json.load(f)
print(metadata)

{'num_cols': ['length', 'diameter', 'height', 'whole_weight', 'shucked_weight', 'viscera_weight', 'shell_weight'], 'cat_cols': ['sex'], 'target_col': 'rings'}


In [4]:
target_column = metadata["target_col"]
x_train = train_df.drop(columns=[target_column]).copy()
y_train = train_df[target_column].to_numpy(dtype=np.float32)

x_test = test_df.drop(columns=[target_column]).copy()
y_test = test_df[target_column].to_numpy(dtype=np.float32)

In [5]:
num_cols = metadata["num_cols"]
cat_cols = metadata["cat_cols"]

preprocessor = ColumnTransformer(
    transformers=[
        ("num", "passthrough", num_cols),
        ("cat", OneHotEncoder(drop="first", handle_unknown="ignore"), cat_cols),
    ],
    remainder="passthrough",
)

In [6]:
best_trial = fit_random_forest_cross_validation(
    X=x_train,
    y=y_train,
    processor=preprocessor,
    task=cfg["task"],
    n_trials=N_TRIALS,
)

rf_model, rf_pipeline, rf_fit_time = fit_best_random_forest(
    X=x_train,
    y=y_train,
    best_trial=best_trial,
    processor=preprocessor,
    task=cfg["task"],
)

[I 2026-04-23 08:02:07,574] A new study created in memory with name: no-name-ce9e272a-1818-42b5-a758-1db2f40eeeb3
[I 2026-04-23 08:02:08,016] Trial 0 finished with value: 4.716048126912144 and parameters: {'n_estimators': 67, 'max_depth': 7, 'min_samples_split': 5, 'min_samples_leaf': 5, 'max_features': 'log2', 'bootstrap': True}. Best is trial 0 with value: 4.716048126912144.
[I 2026-04-23 08:02:08,345] Trial 1 finished with value: 4.645273890999405 and parameters: {'n_estimators': 53, 'max_depth': 8, 'min_samples_split': 8, 'min_samples_leaf': 3, 'max_features': 'log2', 'bootstrap': True}. Best is trial 1 with value: 4.645273890999405.
[I 2026-04-23 08:02:08,653] Trial 2 finished with value: 4.617388682591168 and parameters: {'n_estimators': 43, 'max_depth': 11, 'min_samples_split': 8, 'min_samples_leaf': 4, 'max_features': 'log2', 'bootstrap': True}. Best is trial 2 with value: 4.617388682591168.
[I 2026-04-23 08:02:09,044] Trial 3 finished with value: 4.732964928208519 and paramete

In [13]:
import time
from benchmark.metrics import evaluate_regression


x_test_transformed = rf_pipeline.transform(x_test)

t0 = time.perf_counter()
y_test_pred = rf_model.predict(x_test_transformed)
inference_time = time.perf_counter() - t0

inference_time_per_sample = inference_time / len(x_test)

test_metrics = evaluate_regression(y_test, y_test_pred)


print("Best params:", best_trial.params)
print("Training time (s):", rf_fit_time)
print("Inference time per sample (s):", inference_time_per_sample)
print("Test metrics:", test_metrics)

Best params: {'n_estimators': 73, 'max_depth': 12, 'min_samples_split': 9, 'min_samples_leaf': 4, 'max_features': 'log2', 'bootstrap': True}
Training time (s): 0.0593231380044017
Inference time per sample (s): 4.4283608851105454e-05
Test metrics: {'rmse': 2.1891}


In [14]:
models_dir = MODELS_DIR / data_name
models_dir.mkdir(parents=True, exist_ok=True)

results_dir = RESULTS_BENCHMARK / data_name
results_dir.mkdir(parents=True, exist_ok=True)

joblib.dump(rf_model, models_dir / "random_forest.joblib")
joblib.dump(rf_pipeline, models_dir / "random_forest_pipeline.joblib")

output = {
    **test_metrics,
    "fit_time_s": float(rf_fit_time),
    "infer_time_per_sample_s": float(inference_time_per_sample),
    "best_params": best_trial.params,
}

with open(results_dir / "random_forest.json", "w") as f:
    json.dump(output, f, indent=4)

print(results_dir / "random_forest.json")

/home/shado/college_project/comp9417/9417GroupProject/results/abalone/random_forest.json
